# Guarding the Guardians: Detecting Bias and Underlying Agendas in Environmental Fact-checking Using Machine Learning Approaches 
### Guided BERTopic + machine learning method

Jean Marc Baquero - h12543815

Grégoire Bouvier - h12543344

**Research Question:**
Using machine learning methods to calculate topic density, how does the comparison of the human-verified datasets used to train the Fake News detector algorithms against a diverse reference corpus reveal systematic selection bias in specific contentious environmental sectors?

**Sub-question:**

Sub1: What are the core contentious environmental sectors and underlying socio-economic interests mapped within online climate discourse, and how can they be categorized for algorithmic bias detection?


Sub2: How do advanced embedding and topic modeling techniques (e.g., BERTopic or Transformer representations) compare in quantifying topic density discrepancies between fact-checked data and general media?


Sub3: How can this methodology be be operationalized over time and generalized to offer insights in other important news topics, such as health, politics, technology, etc. to practically surive the continuous introduction of new topics/narratives?

This notebook documents a full pipeline for analyzing climate‑related news using:

- **BERTopic** with guided seed topics  
- **Custom environmental subjects and climate policy taxonomy**  
- **Topic probability heatmaps**  
- **Global News Database (GDELT) keyword tracking** 

The goal is to classify articles into meaningful climate‑policy themes and track how often these themes appear in global news.

**Which datasets / data sources will you work with?**
Misinformation_detection (2018) on GitHub: https://github.com/sfu-discourse-lab/Misinformation_detection/blob/master/README.md
IDD-Resources (2025) on GitHub: https://github.com/mangrue/IDD-Resources
FakeNewsCorpus (2020) on GitHub: https://github.com/several27/FakeNewsCorpus

**Which tools/algorithms/models/metrics do you plan to employ?**
- CorEx (supervised topic modeling): https://github.com/gregversteeg/CorEx
- BERTopic: https://maartengr.github.io/BERTopic/index.html

**Workplan/Milestones:**
- Curate a sub-dataset with environment articles - 29.5.2026.
- Sub-dataset from 3 datasets – clean-up >JM 
- Use LDA, BERTopic to map subjects of contention – 5.6.2026
- Literature on contentious subjects >BG
- Compare to wider statistics to identify bias/blind-spots - 5.6.2026.
- Implement the model and map the topic density discrepancies on fact-checkers VS general media - 12.6.2026.
- Create Tags and add them to the dataset - 19.6.2026
- Generalization for other topics and other contextual flags - 26.6.2026.



### Query GDELT for multiple keywords and append results (keyword | count)

as an example, we will try these keywords = [
    "electric",
    "gas",
    "nuclear",
    "EV"
]

In [2]:
import requests
import pandas as pd

import urllib.parse
import time

def gdelt_keyword_counts(keywords, retries=2, delay=1):
   
    results = []

    for keyword in keywords:
        encoded_keyword = urllib.parse.quote(keyword)
        url = (
            "https://api.gdeltproject.org/api/v2/doc/doc"
            f"?query={encoded_keyword}&mode=TimelineVol&format=json"
        )

        total_count = 0
        success = False

        for attempt in range(retries):
            try:
                req = requests.get(url, timeout=10)

                # Empty responses
                if req.status_code != 200 or not req.text.strip().startswith("{"):
                    raise ValueError("Error: Empty response")

                data = req.json()

                if "timeline" not in data or len(data["timeline"]) == 0:
                    raise ValueError("Error: No timeline data")

                df = pd.DataFrame(data["timeline"])

                total_count = int(df["value"].sum())
                success = True
                break

            except Exception as e:
                print(f"Error: '{keyword}' failed (attempt {attempt+1}/{retries}): {e}")
                time.sleep(delay)

        if not success:
            print(f"Error: No valid data for '{keyword}'.")

        results.append({"keyword": keyword, "count": total_count})

    return pd.DataFrame(results)


keywords = [
    "electric",
    "gas",
    "nuclear",
    "EV"
]

df = gdelt_keyword_counts(keywords)
print(df)

print(f"Done!")


Error: 'electric' failed (attempt 1/2): Error: Empty response
Error: 'electric' failed (attempt 2/2): Error: Empty response
Error: No valid data for 'electric'.
Error: 'gas' failed (attempt 1/2): Error: Empty response
Error: 'gas' failed (attempt 2/2): Error: Empty response
Error: No valid data for 'gas'.
Error: 'nuclear' failed (attempt 1/2): Error: Empty response
Error: 'nuclear' failed (attempt 2/2): Error: Empty response
Error: No valid data for 'nuclear'.
Error: 'EV' failed (attempt 1/2): Error: Empty response
Error: 'EV' failed (attempt 2/2): Error: Empty response
Error: No valid data for 'EV'.
    keyword  count
0  electric      0
1       gas      0
2   nuclear      0
3        EV      0
Done!


# Conclusion

In this notebook, we:

- Built a guided BERTopic model using climate‑policy seed topics  
- Classified news articles into meaningful themes  
- Generated a topic probability heatmap  
- Queried GDELT for keyword frequency
